In [1]:
print("Kernel works")

Kernel works


In [2]:
import pandas as pd
from rdkit import Chem, RDLogger

# Hide RDKit informational and warning messages
RDLogger.DisableLog("rdApp.*")

df = pd.read_csv('/home/susan/mof-co2-adsorption/data/processed/df_chem.csv')
print(df.columns)
print("Dataset shape:", df.shape)
print("Missing MOFID:", df["mofid"].isna().sum())
print("Unique MOFID:", df["mofid"].nunique())


Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='object')
Dataset shape: (27706, 11)
Missing MOFID: 0
Unique MOFID: 24958


In [3]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


---
we want to answer three questions:

- How many extra occurrences are caused by repeated MOFIDs?
- How many different MOFID strings appear more than once?
- How many times can one MOFID occur?

Step 1 — Count occurrence of each MOFID.


In [4]:
mofid_counts = df["mofid"].value_counts()
# Keep only MOFIDs that appear more than once
repeated_mofids = mofid_counts[mofid_counts > 1]

print("Total rows:", len(df))
print("Unique MOFIDs:", df["mofid"].nunique())
print("Repeated occurrences:", df["mofid"].duplicated().sum())

print(mofid_counts.head())


Total rows: 27706
Unique MOFIDs: 24958
Repeated occurrences: 2748
mofid
* MOFid-v1.NA.NA                                                                         1259
* MOFid-v1.NA.NAno_mof                                                                    495
N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERROR.cat0                                     39
N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERROR.cat0                                     32
[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21
Name: count, dtype: int64


| Output                                | Meaning                                                                                                                                             |
| ------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Total rows: 27,706**                |  dataset contains 27,706 MOF records with a non-null value in the `mofid` column.                                                               |
| **Unique MOFIDs: 24,958**             | There are 24,958 different `mofid` strings among those 27,706 rows.                                                                                 |
| **Extra repeated occurrences: 2,748** | After keeping the first occurrence of every MOFID, there are 2,748 additional occurrences of already-seen MOFID strings. This is `27,706 − 24,958`. |
| **Unique MOFIDs that repeat: 616**    | There are 616 different MOFID strings that occur at least twice.                                                                                    |
| **Maximum occurrence: 1,259**         | The most frequently occurring MOFID string appears in 1,259 rows.                                                                                   |


Total rows: 27706
Unique MOFIDs: 24958
Extra repeated occurrences: 2748
Unique MOFIDs that repeat: 616 `number of repeated MOFID types`
Maximum occurrence of one MOFID: 1259 `highest frequency of one MOFID type`
mofid

---


`* MOFid-v1.NA.NA    1259`
means this exact string occurs in 1,259 rows. NA indicates that a normal MOFID representation was not available/generated, so these are not 1,259 copies of the same chemical structure.

---
*MOFid-v1.NA.NAno_mof   495*
means this exact status-like string occurs 495 times. Again, this is not a normal chemical MOFID.

---

*N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERRORcat0  39*

means the exact same MOFID string occurs 39 times. Unlike the NA entries, it contains chemical fragments (N#N, linker, Zn) but its MOFID metadata contains ERROR.

---

*N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERRORcat0   32*
is another specific chemical representation occurring 32 times, also carrying an ERROR status.

---
*[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21*
occurs 21 times. This looks different because pcu is a topology designation rather than an NA or ERROR marker.

*The key distinction is:*
27,706 rows ≠ 27,706 unique MOFIDs. There are 24,958 unique MOFID strings, and the repetition is strongly influenced by placeholder/error values such as the 1,259 NA records.

removing both
* MOFid-v1.NA.NA
          ^^^^^^^^^^

* MOFid-v1.NA.NAno_mof
          ^^^^^^^^^^

In [5]:
# removing both
# MOFid-v1.NA.NA
#  MOFid-v1.NA.NAno_mof
df = df[
    ~df["mofid"].str.contains("MOFid-v1.NA", na=False)
].copy()

print("Remaining rows:", len(df))
#Then verify:
print(df.shape)
print(df["mofid"].str.contains("MOFid-v1.NA", na=False).sum())


Remaining rows: 25952
(25952, 11)
0


In [6]:
# quantify only the ERROR records:
error_count = df["mofid"].str.contains(
    "MOFid-v1.ERROR", na=False
).sum()

print("ERROR-type MOFIDs:", error_count)

# take Take one ERROR MOFID to check it works with rdkit
error_example = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"].iloc[0]

print(error_example)

# extract it :
chemical_smiles = error_example.split(" ")[0]
print(chemical_smiles)

# Test with RDKIT 
mol = Chem.MolFromSmiles(chemical_smiles)
print(mol)

ERROR-type MOFIDs: 1691
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.ERROR
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn]
None


In [7]:
# Now test all 1,691 ERROR records with RDkit : 
error_mofids = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"]

parsed = 0
failed = 0

for mofid in error_mofids:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
    else:
        parsed += 1

print("Total ERROR records:", len(error_mofids))
print("Successfully parsed:", parsed)
print("Failed to parse:", failed)

Total ERROR records: 1691
Successfully parsed: 1628
Failed to parse: 63


---

*27,706 non-null MOFID rows → remove 1,754 NA placeholders → 25,952 rows → 1,691 ERROR-labelled records → 1,628 parse successfully and only 63 fail.*

---




### Can RDKit parse the chemical representation for all 25,952 remaining records?

In [8]:
print("Dataset shape:", df.shape)

Dataset shape: (25952, 11)


In [9]:
# first look at one MOFID from  dataset:
mofid = df["mofid"].iloc[0]

print(mofid)

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0


---
I has two part chemical representation and MOFid metatdata:  `MOFid-v1.pcu.cat0`

---

In [10]:
# .split(" ") separates the MOFID string at the space.
# [0] selects the chemical representation before the MOFID metadata.
chemical_smiles = mofid.split(" ")[0]

print(chemical_smiles)

from rdkit import Chem

mol = Chem.MolFromSmiles(chemical_smiles)

print(mol)

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]
None


----
whether we can isolate the linker from your MOFID and process that linker with RDKit.

----

In [11]:
fragments = chemical_smiles.split(".")

print(fragments)
fragment = Chem.MolFromSmiles(fragments[0])

print(fragment)

['[O-]C(=O)c1ccc(cc1)C(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']


---
<rdkit.Chem.rdchem.Mol object at 0x75c18c10e8f0>

RDKit successfully parsed the first fragment.

---

In [12]:
# Check whole dataset with RDKit
failed_mofids = []
parsed = 0  # number RDKit successfully reads
failed = 0  # number RDKit cannot read
for mofid in df["mofid"]:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
        failed_mofids.append(mofid)
    else:
        parsed += 1

print("Successfully parsed:", parsed)
print("Failed to parse:", failed)


Successfully parsed: 12940
Failed to parse: 13012


---
Can standard RDKit parse all 25,952 complete chemical representations?

No. It parses 12,940, while 13,012 fail.

Next Step: should be only to collect the failed MOFIDs so we can inspect what causes these failures.

----

In [13]:
# collect failed mofid to understand about their features:
# We check a small batch of dataset
for mofid in failed_mofids[:10]:
    print(mofid)
    print()

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1C(=O)[O-])C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=O)C#CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

[O-]C(=O)c1cc(Br)c2c(c1)ccc(c2)C(=O)[O-].[O-]C(=O)C(=O)[O-].[O-]C(=O)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.UNKNOWN.cat0

[O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2)C(=O)[O-])C.[O-]C(=O)c1ccc2c(c1C)c(C)c(c(c2C)C(=O)[O-])C.[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2)C(=O)[O-])C.[O-]C(=O)c1ccc2c(c1C)c(C)c(c(c2C)C(=O)[O-])C.[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

-----

This supports hypothesis that RDKit may be rejecting the Zn–O coordination fragment, while the organic linker itself may still be readable.

-----

In [14]:
# How many of the 13,012 failed MOFIDs contain this exact Zn–O fragment?
# How many of the 13,012 failed MOFIDs contain this exact Zn–O fragment?

zn_o_cluster = "[Zn][O]([Zn])([Zn])[Zn]"

with_zn_o_cluster = 0
without_zn_o_cluster = 0

for mofid in failed_mofids:
    if zn_o_cluster in mofid:
        with_zn_o_cluster += 1
    else:
        without_zn_o_cluster += 1

print(
    f"With Zn-O cluster: {with_zn_o_cluster}, "
    f"Without Zn-O cluster: {without_zn_o_cluster}"
)

With Zn-O cluster: 12284, Without Zn-O cluster: 728


In [15]:
without_zn_o_cluster = []
for mofid in failed_mofids:
    if zn_o_cluster not in mofid:
        without_zn_o_cluster.append(mofid)

for mofid in without_zn_o_cluster[:10]:
    print(mofid)
    print() 

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1cc(O)c(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1cc(O)c(cc1O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat1

CCOC1=[N]=C(C=N[CH]1)OCC.CCOc1cc(cc(c1C(=O)[O-])OCC)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

CCOC1=[N]=C(C(=N[CH]1)OCC)OCC.CCOc1cc(C(=O)[O-])c(cc1C(=O)[O-])OCC.[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

CCOC1=[N]=C(C(=N[CH]1)OCC)OCC.CCOc1cc(C(=O)[O-])c(cc1C(=O)[O-])OCC.[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1ccc(cc1)C(=O)[O-].CCCOC1=[N]=C(C(=N[CH]1)OCCC)OCCC.[O-]C(=O)c1ccc(cc1OCCC)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

COC1=[N]=C(OC)C=C([CH]1)c1ccncc1OC.COc1cc(C(=O)[O-]

In [16]:
# get overall overview of data :
error_count = 0
pcu_cat0_count = 0
pcu_cat1_count = 0
n2_count = 0

for mofid in failed_mofids:
    if "ERROR" in mofid:
        error_count += 1
    if "pcu.cat0" in mofid:
        pcu_cat0_count += 1
    if "pcu.cat1" in mofid:
        pcu_cat1_count += 1
    if "N#N" in mofid:
        n2_count += 1

print("ERROR:", error_count)
print("pcu.cat0:", pcu_cat0_count)
print("pcu.cat1:", pcu_cat1_count)
print("Contains N#N:", n2_count)

ERROR: 63
pcu.cat0: 7210
pcu.cat1: 3659
Contains N#N: 6


| Pattern        | Failed MOFIDs |
| -------------- | ------------: |
| `ERROR`        |            63 |
| `pcu.cat0`     |         7,210 |
| `pcu.cat1`     |         3,659 |
| Contains `N#N` |             6 |


In [17]:
# fragment analysis and Linker extraction:
# Step 1 :Extract the chemical portion of each MOFID by removing the MOFid-v1... metadata.
chemical_representations = []

for mofid in df["mofid"]:
    chemical_smiles = mofid.split("MOFid-v1")[0].strip() # space and mofide_v1“For every MOFID, remove the MOFID metadata and store only its chemical representation.”
    chemical_representations.append(chemical_smiles)

for chemical in chemical_representations[:5]:
    print(chemical)
    print()


[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1C(=O)[O-])C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=O)C#CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]



In [18]:
df["chemical_representation"] = chemical_representations
print(df.columns)
df_fragments = df.copy()

Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg', 'chemical_representation'],
      dtype='object')


In [19]:
# Metal Discconector
from rdkit.Chem.MolStandardize import rdMolStandardize
chemical = df["chemical_representation"].iloc[0]
# Create RDKit molecule without standard sanitization
mof_mol = Chem.MolFromSmiles(chemical, sanitize=False)
mof_mol.UpdatePropertyCache(strict=False)
print(mof_mol)
# Initialize MetalDisconnector
disconnector = rdMolStandardize.MetalDisconnector()
# Disconnect metal-nonmetal bonds
disconnected_mol = disconnector.Disconnect(mof_mol)


In [20]:
# Split the disconnected MOF into individual molecular fragments
fragments = Chem.GetMolFrags(disconnected_mol, asMols=True)

# Print the total number of fragments obtained
print(len(fragments))

# Convert each RDKit fragment back to a SMILES string and display it
for fragment in fragments:
    print(Chem.MolToSmiles(fragment))


6
O=C([O-])c1ccc(C(=O)[O-])cc1
[Zn+]
[O-2]
[Zn+]
[Zn]
[Zn]


In [21]:
# Define the atomic numbers that correspond to metals
metal_atomic_numbers = (
    list(range(3, 5)) +
    list(range(11, 14)) +
    list(range(19, 32)) +
    list(range(37, 51)) +
    list(range(55, 85)) +
    list(range(87, 113))
)
# Define SMARTS for unwanted inorganic nodes/solvents
inorganic_blacklist = [
    Chem.MolFromSmarts('[O-2]'),       # Oxide ion bridges
    Chem.MolFromSmarts('N#N'),         # Nitrogen gas
    Chem.MolFromSmarts('[O;H2]'),       # Water
    Chem.MolFromSmarts('[O-]S(=O)(=O)[O-]') # Sulfate (if applicable)
]
# Create an empty list for fragments that are not metal atoms
non_metal_fragments = []

# Examine each fragment from the first MOF
for frag in fragments:

    # If the fragment contains exactly one atom
    if frag.GetNumAtoms() == 1:

        # Get that atom
        atom = frag.GetAtomWithIdx(0)

        # If that atom is a metal, skip it
        if atom.GetAtomicNum() in metal_atomic_numbers:
            continue
      # Check if the fragment matches any blacklisted substructure completely
    is_inorganic = False
    for pattern in inorganic_blacklist:
        if frag.HasSubstructMatch(pattern):
            # Ensure the match isn't just a part of a larger organic molecule
            if frag.GetNumHeavyAtoms() == pattern.GetNumHeavyAtoms():
                is_inorganic = True
                break
                
    if not is_inorganic:
    # Keep everything that was not removed as a metal
        non_metal_fragments.append(frag)

# Display the fragments remaining after metal removal
for frag in non_metal_fragments:
    print(Chem.MolToSmiles(frag))

O=C([O-])c1ccc(C(=O)[O-])cc1


In [22]:

# Define the atomic numbers that correspond to metals
metal_atomic_numbers = (
    list(range(3, 5)) +
    list(range(11, 14)) +
    list(range(19, 32)) +
    list(range(37, 51)) +
    list(range(55, 85)) +
    list(range(87, 113))
)

# Define SMARTS for unwanted inorganic nodes/solvents
inorganic_blacklist = [
    Chem.MolFromSmarts('[O-2]'),
    Chem.MolFromSmarts('N#N'),
    Chem.MolFromSmarts('[O;H2]'),
    Chem.MolFromSmarts('[O-]S(=O)(=O)[O-]')
]

def extract_linkers(chemical):

    # Read the MOF representation without sanitizing
    lig_mol = Chem.MolFromSmiles(chemical, sanitize=False)

    if lig_mol is None:
        raise ValueError(
            "RDKit could not create a molecule from the chemical representation"
        )

    lig_mol.UpdatePropertyCache(strict=False)

    # Disconnect metal-ligand bonds
    disconnector = rdMolStandardize.MetalDisconnector()
    disconnected_mol = disconnector.Disconnect(lig_mol)

    # Split the disconnected structure into individual fragments
    fragments = Chem.GetMolFrags(
        disconnected_mol,
        asMols=True,
        sanitizeFrags=False
    )

    linker_smiles = []
    invalid_fragments = []

    for frag in fragments:

        # Remove isolated metal atoms
        if frag.GetNumAtoms() == 1:
            atom = frag.GetAtomWithIdx(0)

            if atom.GetAtomicNum() in metal_atomic_numbers:
                continue

        # Remove known small inorganic fragments
        is_inorganic = any(
            frag.HasSubstructMatch(pattern)
            and frag.GetNumHeavyAtoms() == pattern.GetNumHeavyAtoms()
            for pattern in inorganic_blacklist
        )

        if is_inorganic:
            continue

        # Now sanitize ONLY this candidate fragment
        try:
            frag.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(frag)

            linker_smiles.append(
                Chem.MolToSmiles(frag)
            )

        except Exception as error:
            invalid_fragments.append({
                "fragment": Chem.MolToSmiles(frag),
                "error": str(error)
            })

    return linker_smiles, invalid_fragments


In [23]:
extract_linkers(df["chemical_representation"].iloc[0])

(['O=C([O-])c1ccc(C(=O)[O-])cc1'], [])

In [24]:
# Create an empty list to store the extracted linker(s) for every MOF
all_linkers = []

# Create an empty list to record MOFs where linker extraction fails
failed_linker_extraction = []

# Loop through each row using the DataFrame index and chemical representation
for index, chemical in df["chemical_representation"].items():

    # Try to extract the linker(s) from the current MOF
    try:

        # Apply our extract_linkers function to the current chemical representation
        linkers = extract_linkers(chemical)

        # Store the extracted linker(s) in the results list
        all_linkers.append(linkers)

    # If RDKit or another operation produces an error, catch it here
    except Exception as error:

        # Print the DataFrame row index where extraction failed
        print(f"Failed at row {index}: {error}")

        # Print the chemical representation that caused the error
        print(f"Chemical representation: {chemical}")

        # Add an empty list so this row still has a corresponding result
        all_linkers.append([])

        # Store detailed information about the failed MOF for later investigation
        failed_linker_extraction.append(
            {
                # Store the original DataFrame index
                "index": index,

                # Store the chemical representation that failed
                "chemical_representation": chemical,

                # Convert the error message to text and store it
                "error": str(error),
            }
        )

# Add the extracted linker lists to the original DataFrame as a new column
df["linker_smiles"] = all_linkers

# Print the total number of MOFs where linker extraction failed
print("Failed extractions:", len(failed_linker_extraction))

Failed at row 5616: Explicit valence for atom # 118 Br, 2, is greater than permitted
Chemical representation: [O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].[O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].BrOC(=O)c1cc(Br)c(c(c1Br)Br)C(=O)[O-].BrOC(=O)C1=[C]C(=C(C=C1)c1ccc(cc1[Br]12[O]3[C]4O[Zn]56[O]78[Zn]93[O]1[C]1c3ccc(c(c3)Br)C3=C([C]=C([C](O5)[O]([Zn]57O[C]([O]29)c2c(Br)cc(c(c2Br)Br)[C](O6)[O]2[Zn]68([O]1[Br]c1c(C(=O)[O-])c(Br)cc(c1Br)C(=O)OBr)[O]([Br]26)[C](O5)C1=C(C=C4C(=[C]1)Br)Br)Br)C=C3)Br)C(=O)[O-])Br.BrOC(=O)C1=[C]C(=C(C=C1)c1ccc(cc1Br)C(=O)[O-])Br.[Zn][O]([Zn])([Zn])[Zn]
Failed at row 11070: Explicit valence for atom # 12 O, 3, is greater than permitted
Chemical representation: O=C1C#Cc2ccc3cc2[C]2[N]45[O]62[Zn]27895[NH]4N4[C]5[N]%109[O]8[C]8O[Zn]9%114O[C]6C#Cc4ccc(C#C[C](O7)O%11)c6c4c([N])n4c6[N]67[O]%11[C]%12C#Cc%13cc%14[C]%15[O]%16%17[Zn]%18%19%20(O1)(OC(=O)C#Cc1ccc(C#CC%17=O)c%17c1c%10n5c%17[N])[NH]([N]%15%16%20)[N][C]1[N]%19([O]%18C(=O)C#C3)c3n1c(c1c3c3C#C[C]5O[Zn]%10%15(O[C](C#Cc

In [25]:
df[["mofid", "chemical_representation", "linker_smiles"]].head(10)

,mofid,chemical_representation,linker_smiles
0,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,"([O=C([O-])c1ccc(C(=O)[O-])cc1], [])"
1,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,"([O=C([O-])c1ccc(C(=O)[O-])cc1], [])"
2,[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Z...,[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Z...,"([O=C([O-])c1cc(F)c(C(=O)[O-])c(F)c1F], [])"
3,COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1...,COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1...,"([COc1cc(C(=O)[O-])cc(OC)c1C(=O)[O-], COc1cc(C..."
4,CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=...,CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=...,"([CCc1cc(C(=O)[O-])c(CC)c(CC)c1C(=O)[O-], O=C(..."
5,[O-]C(=O)c1cc(Br)c2c(c1)ccc(c2)C(=O)[O-].[O-]C...,[O-]C(=O)c1cc(Br)c2c(c1)ccc(c2)C(=O)[O-].[O-]C...,"([O=C([O-])c1cc(Br)c2cc(C(=O)[O-])ccc2c1, O=C(..."
6,[O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C...,[O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C...,"([O=C([O-])C(=O)[O-], Cc1c(C(=O)[O-])ccc2cc(C(..."
7,[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]...,[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]...,"([Cc1c(C(=O)[O-])cc2ccc(C(=O)[O-])c(C)c2c1C, C..."
8,[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]...,[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]...,"([Cc1c(C(=O)[O-])cc2ccc(C(=O)[O-])c(C)c2c1C, C..."
9,CCc1cc(C(=O)[O-])c(c(c1[C][O])CC)CC.[O].[O-]C(...,CCc1cc(C(=O)[O-])c(c(c1[C][O])CC)CC.[O].[O-]C(...,"([CCc1cc(C(=O)[O-])c(CC)c(CC)c1[C][O], [O], O=..."


In [26]:
df["linker_smiles"].apply(len).value_counts().sort_index()

linker_smiles
0        4
2    25948
Name: count, dtype: int64

In [27]:
# Select only MOFs where no linker was extracted
failed_linkers_df = df[df["linker_smiles"].apply(len) == 0].copy()

# Check the number of failed MOFs
print(failed_linkers_df.shape)

# Inspect their chemical representations
failed_linkers_df[
    ["mofid", "chemical_representation", "linker_smiles"]
].head(20)

(4, 13)


,mofid,chemical_representation,linker_smiles
5616,[O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].[O-]...,[O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].[O-]...,[]
11070,O=C1C#Cc2ccc3cc2[C]2[N]45[O]62[Zn]27895[NH]4N4...,O=C1C#Cc2ccc3cc2[C]2[N]45[O]62[Zn]27895[NH]4N4...,[]
24142,BrC1=C2[C]=CC(=C1Br)c1ccc(c(c1)[Br]13[O]4[C]5O...,BrC1=C2[C]=CC(=C1Br)c1ccc(c(c1)[Br]13[O]4[C]5O...,[]
27218,[O-]C(=O)[C]=C([Br]1[O]2C3=C[CH]C(=[C][C]4O[Zn...,[O-]C(=O)[C]=C([Br]1[O]2C3=C[CH]C(=[C][C]4O[Zn...,[]


In [28]:
# Convert the stored failure information into a DataFrame
failed_df = pd.DataFrame(failed_linker_extraction)

# Check the available diagnostic columns
print(failed_df.columns)

# Display the first few failure records
failed_df.head()

Index(['index', 'chemical_representation', 'error'], dtype='object')


,index,chemical_representation,error
0,5616,[O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].[O-]...,"Explicit valence for atom # 118 Br, 2, is grea..."
1,11070,O=C1C#Cc2ccc3cc2[C]2[N]45[O]62[Zn]27895[NH]4N4...,"Explicit valence for atom # 12 O, 3, is greate..."
2,24142,BrC1=C2[C]=CC(=C1Br)c1ccc(c(c1)[Br]13[O]4[C]5O...,"Explicit valence for atom # 41 O, 3, is greate..."
3,27218,[O-]C(=O)[C]=C([Br]1[O]2C3=C[CH]C(=[C][C]4O[Zn...,"Explicit valence for atom # 17 O, 3, is greate..."


In [29]:
# Classify the RDKit errors by type
def classify_error(error):
    if "Explicit valence" in error and " N," in error:
        return "N valence"
    elif "Explicit valence" in error and " O," in error:
        return "O valence"
    elif "Explicit valence" in error:
        return "Other valence"
    else:
        return "Other error"

failed_df["error_type"] = failed_df["error"].apply(classify_error)

failed_df["error_type"].value_counts()

error_type
O valence        3
Other valence    1
Name: count, dtype: int64

In [30]:
# Display the complete error messages for the other-valence failures
failed_df.loc[
    failed_df["error_type"] == "Other valence",
    "error"
].value_counts()


error
Explicit valence for atom # 118 Br, 2, is greater than permitted    1
Name: count, dtype: int64

## Note: 

Linker extraction succeeded for 25,948 of 25,952 MOFs (99.98%). Four structures were excluded from chemistry-descriptor generation because their MOFid-derived representations produced unresolved RDKit explicit-valence errors after metal disconnection (three O-valence and one Br-valence case). No manual charge or bonding corrections were applied to avoid altering the reported chemical structures.

In [33]:
#Remove only the 4 failed MOFs
failed_final_df = df[df["linker_smiles"].apply(len) == 0].copy()
print(failed_final_df['filename'])

5616       hMOF-151
11070    hMOF-20326
24142      hMOF-368
27218     hMOF-6556
Name: filename, dtype: object


In [34]:
# create a new DataFrame containing only successfully processed MOFs
# Keep only MOFs with successfully extracted linkers
df_chemistry = df[df["linker_smiles"].apply(len) > 0].copy()

# Check the new dataset size
print("Original Phase 2 dataset:", df.shape)
print("Final chemistry dataset:", df_chemistry.shape)

Original Phase 2 dataset: (25952, 13)
Final chemistry dataset: (25948, 13)


In [38]:
print(df_chemistry['linker_smiles'])
# Save the final linker-extracted dataset
df_chemistry.to_csv(
    "/home/susan/mof-co2-adsorption/data/processed/hmof_linker_extracted.csv",
    index=False
)

0                     ([O=C([O-])c1ccc(C(=O)[O-])cc1], [])
1                     ([O=C([O-])c1ccc(C(=O)[O-])cc1], [])
2              ([O=C([O-])c1cc(F)c(C(=O)[O-])c(F)c1F], [])
3        ([COc1cc(C(=O)[O-])cc(OC)c1C(=O)[O-], COc1cc(C...
4        ([CCc1cc(C(=O)[O-])c(CC)c(CC)c1C(=O)[O-], O=C(...
                               ...                        
27701                ([Nc1cc(C(=O)[O-])ccc1C(=O)[O-]], [])
27702    ([Cc1cc(C(=O)[O-])c2c(c1C(=O)[O-])C(C)C2(C)C, ...
27703    ([O=C([O-])C#CC(=O)[O-], O=C([O-])C(Cl)=CC(Cl)...
27704    ([O=C([O-])C#CC(=O)[O-], O=C([O-])C(Cl)=CC(Cl)...
27705           ([O=C([O-])[C]=CC(Cl)=C(Cl)C(=O)[O-]], [])
Name: linker_smiles, Length: 25948, dtype: object
